In [ ]:
from skimage.feature import graycomatrix, graycoprops
from skimage.color import rgb2gray
from skimage import measure
from sklearn.preprocessing import normalize
import numpy as np
import cv2
from PIL import Image


In [ ]:
# 1. Función para calcular la distribución de colores (histograma)
def extract_color_histogram(image, bins=(8, 8, 8)):
    """
    Extrae un histograma de color en formato RGB.
    """
    # Convertir a array NumPy
    image_array = np.array(image)

    # Calcular histograma en RGB
    hist = cv2.calcHist([image_array], [0, 1, 2], None, bins, [0, 256, 0, 256, 0, 256])

    # Normalizar el histograma
    hist = cv2.normalize(hist, hist).flatten()
    return hist

In [ ]:
# 2. Función para extraer características de textura usando GLCM
def extract_texture_features(image, distances=[1], angles=[0]):
    """
    Extrae características de textura utilizando la Matriz de Co-ocurrencia de Niveles de Gris (GLCM).
    """
    # Convertir a escala de grises
    gray_image = rgb2gray(np.array(image))
    gray_image = (gray_image * 255).astype(np.uint8)  # Escalar a 0-255

    # Calcular GLCM
    glcm = graycomatrix(gray_image, distances=distances, angles=angles, levels=256, symmetric=True, normed=True)

    # Extraer propiedades de textura
    contrast = graycoprops(glcm, 'contrast')[0, 0]
    dissimilarity = graycoprops(glcm, 'dissimilarity')[0, 0]
    homogeneity = graycoprops(glcm, 'homogeneity')[0, 0]
    energy = graycoprops(glcm, 'energy')[0, 0]
    correlation = graycoprops(glcm, 'correlation')[0, 0]

    return np.array([contrast, dissimilarity, homogeneity, energy, correlation])

In [ ]:
# 3. Función para extraer características de forma
def extract_shape_features(image):
    """
    Extrae características de forma utilizando contornos y propiedades geométricas.
    """
    # Convertir a escala de grises y binarizar
    gray_image = rgb2gray(np.array(image))
    binary_image = (gray_image > 0.5).astype(np.uint8)

    # Encontrar contornos
    contours = measure.find_contours(binary_image, 0.5)

    # Calcular características geométricas
    if contours:
        largest_contour = max(contours, key=lambda x: measure.perimeter(x))
        area = measure.mesh_surface_area(largest_contour)
        perimeter = measure.perimeter(largest_contour)
        eccentricity = measure.regionprops_table(binary_image, properties=['eccentricity'])['eccentricity'][0]
        return np.array([area, perimeter, eccentricity])
    else:
        return np.array([0, 0, 0])  # Si no hay contornos

In [ ]:
# 4. Función para extraer todas las características
def extract_features(image):
    """
    Extrae todas las características (color, textura, forma) de una imagen procesada.
    """
    color_features = extract_color_histogram(image)
    texture_features = extract_texture_features(image)
    shape_features = extract_shape_features(image)

    # Concatenar todas las características
    all_features = np.concatenate([color_features, texture_features, shape_features])
    return normalize(all_features.reshape(1, -1)).flatten()  # Normalizar

In [ ]:
# 5. Función principal para procesar un conjunto de imágenes
def process_features(input_folder, output_file="features.csv"):
    """
    Procesa todas las imágenes en un directorio para extraer características y guardarlas en un archivo CSV.
    """
    import csv

    feature_list = []
    headers = [
        *["color_bin_" + str(i) for i in range(8 * 8 * 8)],
        "contrast", "dissimilarity", "homogeneity", "energy", "correlation",
        "area", "perimeter", "eccentricity"
    ]

    for filename in os.listdir(input_folder):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            image_path = os.path.join(input_folder, filename)
            print(f"Extrayendo características de: {filename}")

            try:
                image = Image.open(image_path)
                features = extract_features(image)
                feature_list.append([filename] + features.tolist())
            except Exception as e:
                print(f"Error al procesar {filename}: {e}")
                continue

    # Guardar las características en un archivo CSV
    with open(output_file, mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(["filename"] + headers)  # Encabezados
        writer.writerows(feature_list)
    print(f"Características guardadas en: {output_file}")

In [ ]:
# Ejemplo de uso
if __name__ == "__main__":
    input_folder = "/ruta/a/imagenes/procesadas"  # Cambia por la carpeta de imágenes procesadas
    output_file = "features.csv"  # Archivo de salida con las características
    process_features(input_folder, output_file)
